In [ ]:
import pandas as pd
import numpy as np 



In [ ]:
df=pd.read_csv("raw_car_dataset.csv")

In [ ]:
print(df.head(10))

In [ ]:
print(df.shape)

In [ ]:
print(df.info())

In [ ]:
print("missing values:")
print(df.isnull().sum())

In [ ]:
print("show the counts of location:")
print(df["Location"].value_counts(dropna=False))

In [ ]:
print(" clean section mareyna $ iyo ,")
df["Price"]=df["Price"].replace(r"[\$,]","", regex=True).astype(float)
print(df["Price"].head(10))

In [ ]:
print(df["Price"].skew())


3-Fix CATEGORY LOCATION

In [ ]:

df["Location"]=df["Location"].replace({"Subrb": "Suburb", "??": pd.NA})
print("laction counts after fixed type/unknown")
print(df["Location"].value_counts(dropna=False))

4-Impute Missing Values (justify choices)


In [ ]:
df["Odometer_km"]=df["Odometer_km"].fillna(df["Odometer_km"].median())
df["Doors"]=df["Doors"].fillna(df["Doors"].mode()[0])
df["Accidents"]=df["Accidents"].fillna(df["Accidents"].mode()[0])
df["Location"]=df["Location"].fillna(df["Location"].mode()[0])
print(df.isnull().sum())

5-Remove Duplicates

In [ ]:
bofore=df.shape

In [ ]:
df=df.drop_duplicates()

In [ ]:
after=df.shape

In [ ]:
print("before: ", bofore,"After: ",after)

7-Outliers (IQR capping)

In [ ]:
def iqr_fun(series,k=1.5):
    q1,q3=series.quantile([0.25,0.75])
    iqr = q3 - q1
    lower = q1 - k * iqr
    upper = q3 + k * iqr
    return lower, upper
low_price, high_price=iqr_fun(df["Price"])
low_odometer, high_odometer=iqr_fun(df["Odometer_km"])

In [ ]:
print("IQR Of Price : " )
print("Low_Price: ", low_price , "High_Price: " , high_price)

In [ ]:
print("IQR Of Odometer_km : " )
print("Low_Odometer: ", low_odometer , "High_Odometer: " , high_odometer)

after clip

In [ ]:
df.loc[:,"Price"] = df ["Price"].clip(lower = low_price , upper = high_price)
df.loc[:,"Odometer_km"] = df ["Odometer_km"].clip(lower = low_odometer , upper = high_odometer)



In [ ]:
print("The Price after IQR Clipping :")
print(df["Price"].describe())


In [ ]:
print("The Odometer_km after IQR Clipping :")
print(df["Odometer_km"].describe())

7-One-Hot Encode Categorical(s)

In [ ]:


df = pd.get_dummies(
    df,
    columns=["Location"],
    prefix="Location",
    dtype=int
)

dummy_cols = [c for c in df.columns if c.startswith("Location_")]

print("\nSTEP 7: ONE-HOT ENCODING")
print("New dummy columns created:")
print(dummy_cols)

8-Feature Engineering (no leakage)

In [ ]:
CURRENT_YEAR=2026
df["CarAge"]= CURRENT_YEAR-df["Year"]
df["Km_per_year"]= (df["Odometer_km"] / df["CarAge"] + 1)


In [ ]:
df["Is_Urban"] = df["Location_Urban"].astype(int) if "Location_Urban" in df.columns else 0

df["LogPrice"] = np.log1p(df["Price"])

In [ ]:

print(df[["CarAge", "Km_per_year", "Is_Urban", "LogPrice"]].head())

9-Feature Scaling (X only)

In [ ]:
from sklearn.preprocessing import StandardScaler

In [ ]:
from sklearn.preprocessing import StandardScaler

dont_scale = {'Price', 'LogPrice'}
exclude = [c for c in df.columns if c.startswith('Location_')] + ['Is_Urban']
numeric_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
scale_cols = [c for c in numeric_cols if c not in dont_scale and c not in exclude]

df[scale_cols] = StandardScaler().fit_transform(df[scale_cols])
df[scale_cols].head()

10-Final Checks & Save

In [ ]:
print(df.info())

In [ ]:
print("missing values:")
print(df.isnull().sum())

In [ ]:
print("Describe ")
print(df.describe())

In [ ]:
OUT_PATH="car_l3_clean_ready.csv"
df.to_csv(OUT_PATH)
print("Saved to: " , OUT_PATH)